In [ ]:
%pip install --quiet python-dotenv pydantic-ai

import os
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    url = "https://github.com/jsoma/workshop-ai-images-video/raw/main/docs/nicar-2026/01-images-data.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_data.zip")
    with zipfile.ZipFile("_data.zip") as zf:
        zf.extractall("data")
    Path("_data.zip").unlink()
    print("Done!")

# Load API keys: Colab secrets or local .env
try:
    from google.colab import userdata
    os.environ.setdefault("GOOGLE_API_KEY", userdata.get("GOOGLE_API_KEY"))
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# Images

You already know what to do with text: summarize it, answer questions about it, extract data from it. Images, audio, and video are just ways of **getting to text and structured data.**

## Asking questions of images

It's easy enough to send an image to an LLM and ask it some questions. It's easy to read and great for a one-off, but *very* hard to sort or filter across hundreds of images.


In [ ]:
from IPython.display import Image
Image(filename="data/car.jpg", width=500)


Let's get some details about this car.


**`vision-llm/raw-openai-text.py`** — Same task using raw OpenAI client, plain text response — no structured output


In [ ]:
import base64
from pathlib import Path

from openai import OpenAI

MODEL = "gpt-4o-mini"
DATA = Path("data")

client = OpenAI()
base64_image = base64.b64encode((DATA / "car.jpg").read_bytes()).decode("utf-8")

prompt = """List the following about this vehicle:
- make
- model
- type
- color
- estimated year
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": [
        {"type": "text", "text": prompt},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
            }
        },
    ]}],
)
print(response.choices[0].message.content)


Based on the vehicle shown in the image, here are the details:

- **Make:** Toyota
- **Model:** Highlander
- **Type:** SUV
- **Color:** Silver
- **Estimated Year:** Early 2000s (around 2001-2007)


It's great, but... what if we want it in a CSV? What if we have 200 to do? What if we want to make *demands* and ask for *structure?*

## Structured output

An alternative is to send an image to an LLM and get back **structured output** — fields you can sort, filter, and verify. Not prose. This is the pattern for everything else in the workshop!


**`vision-llm/structured.py`** — Send one image to an LLM, get structured Pydantic data back


In [ ]:
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent
from typing import Literal

MODEL = "openai:gpt-4o-mini"
DATA = Path("data")

image_data = (DATA / "car.jpg").read_bytes()

class Vehicle(BaseModel):
    make: str = Field(description="Vehicle manufacturer (e.g., Toyota, Ford)")
    model: str = Field(description="Vehicle model name (e.g., Camry, F-150)")
    color: str = Field(description="Primary color of the vehicle")
    year_estimate: int = Field(description="Estimated model year (best guess)")
    vehicle_type: Literal[
        "sedan", "SUV", "truck", "van", "motorcycle", "other"
    ] = Field(description="Type of vehicle")
    confidence: float = Field(description="Your confidence in this identification, 0.0 to 1.0")

agent = Agent(MODEL, output_type=Vehicle)
result = agent.run_sync([
    "Analyze the vehicle in this image. Fill in all fields accurately.",
    BinaryContent(data=image_data, media_type="image/jpeg"),
])
result.output


Vehicle(make='Toyota', model='Highlander', color='gold', year_estimate=2006, vehicle_type='SUV', confidence=0.9)

While there are a handful of ways to do this, we're specifically using a Python library called **Pydantic**. It gives you a lot of tools to describe what you're looking for: each field has a name, a type, and a description. AI fills in the fields.

Easy peasy!

## Batch processing

Same thing as before, but we have **a whole folder of images**. And instead of one at a time, you can make an entire CSV!


**`vision-llm/batch.py`** — Process a folder of images into structured data -> DataFrame -> CSV


In [ ]:
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent
from typing import Literal

MODEL = "openai:gpt-4o-mini"
DATA = Path("data")

class Vehicle(BaseModel):
    make: str = Field(description="Vehicle manufacturer")
    model: str = Field(description="Vehicle model name")
    color: str = Field(description="Primary color")
    year_estimate: int = Field(description="Estimated model year")
    vehicle_type: Literal[
        "sedan", "SUV", "truck", "van", "motorcycle", "other"
    ] = Field(description="Type of vehicle")
    confidence: float = Field(description="Confidence in identification, 0.0 to 1.0")

agent = Agent(MODEL, output_type=Vehicle)
rows = []
image_paths = sorted((DATA / "cars").glob("*.jpg"))
for image_path in image_paths:
    result = agent.run_sync([
        "Analyze the vehicle in this image. Fill in all fields.",
        BinaryContent(data=image_path.read_bytes(), media_type="image/jpeg"),
    ])
    rows.append({"filename": image_path.name, **result.output.model_dump()})

df = pd.DataFrame(rows)
output = Path("outputs") / "cars_analysis.csv"
output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output, index=False)
df


,filename,make,model,color,year_estimate,vehicle_type,confidence
0,28246634.jpg,Volkswagen,Transporter,black,2018,van,0.85
1,28246768.jpg,Toyota,Camry,silver,2015,sedan,0.95
2,28262472.jpg,Lexus,LX,black,2020,SUV,0.90
3,28262480.jpg,Toyota,Yaris,yellow,2018,sedan,0.90
4,28266737.jpg,Tesla,Model Y,White,2021,SUV,0.90


Open the output CSV. Spot-check a few rows against the source images. Does the make match what you see? Does the color? That's verification — not trusting the model, checking its work.

## Swap providers

There are a ton of different providers of LLM *stuff* and they each have strengths and weaknesses. if you get married to ChatGPT or Claude, you'll never be able to use Gemini's document-processing powers! So instead of using the [genai library from Google](https://github.com/googleapis/python-genai) or the [OpenAI library](https://github.com/openai/openai-python) we use Pydantic AI, which allows you a bit more flexibility in swapping between providers.

If you're feeling especially wild, you can even try out [openrouter](https://openrouter.ai/), which gives you a menu of *way more* than just the Big Three.


**`vision-llm/providers.py`** — Same structured-output task with different LLM providers


In [ ]:
from pathlib import Path

from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")
image_data = (DATA / "sky.jpg").read_bytes()

class ImageDescription(BaseModel):
    subject: str = Field(description="Main subject of the image")
    setting: str = Field(description="Where the image appears to be taken")
    mood: str = Field(description="Overall mood or feeling of the image")
    details: list[str] = Field(description="3-5 notable details")

models = [
    "openai:gpt-4o-mini",
    "google-gla:gemini-2.5-flash",
    # "anthropic:claude-3-5-haiku-latest",
    # "ollama:qwen2-vl",
]
for model in models:
    agent = Agent(model, output_type=ImageDescription)
    result = agent.run_sync([
        "Describe this image. Fill in all fields.",
        BinaryContent(data=image_data, media_type="image/jpeg"),
    ])
    print(f"--- {model} ---")
    print(result.output)


--- openai:gpt-4o-mini ---
subject='A sunset landscape' setting='An open field near a road' mood='Serene and tranquil' details=['The sky is painted in pastel colors, with soft pinks and oranges as the sun sets.', 'In the foreground, there are blooming purple flowers, likely lupines, creating a colorful border.', 'A light mist covers the ground, adding a mystical quality to the scene.', 'There is a single large rock on the left side of the image, contributing to the natural feel.', 'Power lines are visible in the distance along the road, hinting at the rural setting.']


--- google-gla:gemini-2.5-flash ---
subject='A vast landscape with a sunset over a foggy horizon' setting='A rural landscape at sunrise or sunset with a road, fields, and a distant foggy horizon' mood='Serene and peaceful, with a touch of mystery from the fog' details=['A vibrant sunset with orange and yellow hues dominates the right side of the sky.', 'A thick layer of fog or mist blankets the distant horizon and parts of the landscape.', 'Extensive patches of purple flowers are visible in the foreground and along the roadside.', 'A winding road cuts through the landscape, disappearing into the fog.', 'Grassy fields with some darker, bare patches of earth and a large rock in the foreground.']


OpenAI, Google, Anthropic, Ollama — the code is identical except for the model name. Pick whichever fits your newsroom's budget, privacy needs, or existing accounts.
